In [ ]:
from pathlib import Path

import iplotx as ipx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import polars as pl
import polars.selectors as cs
import seaborn as sns
from sklearn.decomposition import PCA

from climate_attitudes import configure_mpl
from climate_attitudes.correlation import Correlation
from climate_attitudes.dataset import Dataset
from climate_attitudes.parallel_analysis import pa_random_eigs, pa_true_eigs
from climate_attitudes.settings import Config

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="ds1_5")

dataset_std = dataset.standardise(cs.exclude("participant_id", "wave"))
resp = dataset.response.collect()
resp_std = dataset_std.response.collect()

In [ ]:
def plot_corr_network(df, corr, threshold: float = 0.05, directed: bool = False):
    fig, ax = plt.subplots(figsize=(15, 15), constrained_layout=True)

    DIVERGING_CMAP = sns.diverging_palette(20, 230, as_cmap=True)

    # Generate a mask for the upper triangle
    if not directed:
        mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    else:
        mask = np.full_like(corr, fill_value=False, dtype=bool)

    # If `mask_below` set, reset square colour below abs value to zero
    mask[abs(corr) < threshold] = True

    # Remove questions where only diagonal is unmasked
    keep_idxes = (2 * mask.shape[1] - (mask.sum(axis=1) + mask.sum(axis=0))) > 2
    corr = corr[keep_idxes][:, keep_idxes]
    mask = mask[keep_idxes][:, keep_idxes]
    node_labels = np.asarray(df.columns)[keep_idxes]

    # ======== Network
    adj = corr
    adj[np.diag_indices_from(adj)] = 0
    adj[mask] = 0
    if directed:
        G = nx.from_numpy_array(adj, create_using=nx.DiGraph)
    else:
        G = nx.from_numpy_array(adj)
    edge_linewidths = {(u, v): z["weight"] * 12 for u, v, z in G.edges(data=True)}
    edge_colours = [z["weight"] for u, v, z in G.edges(data=True)]
    edge_labels = [f"{z['weight']:.2f}" for u, v, z in G.edges(data=True)]
    layout = nx.forceatlas2_layout(G, gravity=1)

    with ipx.style.context(
        [
            "hollow",
            {
                "vertex": {
                    "linewidth": 1,
                },
                "edge": {
                    "color": edge_colours,
                    "alpha": 1,
                    "cmap": DIVERGING_CMAP,
                    "norm": mcolors.Normalize(vmin=-1, vmax=1),
                },
            },
        ]
    ):
        network_artist = ipx.network(
            G,
            layout=layout,
            tension=1,
            edge_labels=edge_labels,
            node_labels=node_labels,
            edge_linewidth=edge_linewidths,
            edge_curved=True,
            # aspect="equal",
            margins=0.1,
            edge_label_bbox=dict(
                edgecolor="black",
                facecolor="white",
                linewidth=0.25,
                boxstyle="round,pad=0.3",
            ),
            edge_label_rotate=True,
            vertex_facecolor="white",
            vertex_zorder=3,
            ax=ax,
        )[0]
        fig.colorbar(
            network_artist.get_edges(),
            shrink=0.6,
            aspect=30,
            ax=ax,
        )

    fig.suptitle("Partial correlation")

In [ ]:
def plot_temporal_network(df, corr, threshold: float = 0.05):
    fig, ax = plt.subplots(figsize=(15, 15), constrained_layout=True)

    DIVERGING_CMAP = sns.diverging_palette(20, 230, as_cmap=True)

    n = corr.shape[0]
    adj = np.zeros((n * 2, n * 2), dtype=np.float64)
    adj[:n, n : 2 * n] = corr

    # # Show self-loops on left side rather than crossing
    # for i in range(n):
    #     adj[i,i] = adj[i,n+i]
    #     adj[i,n+i] = 0.0

    mask = np.full_like(adj, fill_value=False, dtype=bool)

    # If `mask_below` set, reset square colour below abs value to zero
    mask[abs(adj) < threshold] = True

    # Remove questions where only diagonal is unmasked
    # keep_idxes = (2 * mask.shape[1] - (mask.sum(axis=1) + mask.sum(axis=0))) > 2
    # corr = corr[keep_idxes][:, keep_idxes]
    # mask = mask[keep_idxes][:, keep_idxes]
    node_labels = np.asarray([*df.columns] * 2)
    for i in range(n):
        node_labels[i] = f"{node_labels[i]} ({adj[i, n + i]:.2f})"
        adj[i, n + i] = 0.0

    # ======== Network
    # adj = corr
    # adj[np.diag_indices_from(adj)] = 0
    adj[mask] = 0
    G = nx.from_numpy_array(adj, create_using=nx.DiGraph)
    edge_linewidths = {(u, v): z["weight"] * 4 for u, v, z in G.edges(data=True)}
    edge_colours = [z["weight"] for u, v, z in G.edges(data=True)]
    edge_labels = [f"{z['weight']:.2f}" for u, v, z in G.edges(data=True)]
    # layout = nx.forceatlas2_layout(G, gravity=1)
    layout = nx.bipartite_layout(G, nodes=list(G.nodes)[:n])

    with ipx.style.context(
        [
            "hollow",
            {
                "vertex": {
                    "linewidth": 1,
                },
                "edge": {
                    "color": edge_colours,
                    "alpha": 1,
                    "cmap": DIVERGING_CMAP,
                    "norm": mcolors.Normalize(vmin=-1, vmax=1),
                },
            },
        ]
    ):
        _ = ipx.network(
            G,
            layout=layout,
            # tension=1,
            edge_labels=edge_labels,
            node_labels=node_labels,
            edge_linewidth=edge_linewidths,
            # edge_curved=True,
            # aspect="equal",
            margins=0.1,
            edge_label_bbox=dict(
                edgecolor="black",
                facecolor="white",
                linewidth=0.25,
                boxstyle="round,pad=0.3",
            ),
            edge_label_rotate=True,
            vertex_facecolor="white",
            vertex_zorder=3,
            ax=ax,
        )[0]

    fig.suptitle("Partial correlation")

In [ ]:
def betweenness(corr, df, threshold: float = 0.01):
    adj = corr.copy()
    adj[np.diag_indices_from(adj)] = 0
    adj[abs(adj) < threshold] = 0
    G = nx.from_numpy_array(adj)
    for i, j, w in G.edges.data("weight"):
        G.edges[i, j]["dist"] = -abs(w)
    return np.asarray(df.drop("participant_id", "wave").columns)[
        np.argsort(
            np.asarray(list(nx.betweenness_centrality(G, weight="dist").values()))
        )[::-1]
    ]

# Partial correlation

In [ ]:
corr = Correlation.PARTIAL.calculate(resp_std, assume_centred=True)
plot_corr_network(resp_std.drop("participant_id", "wave"), corr)
betweenness(corr, resp_std, threshold=0.05)

# Regularised Partial correlation

In [ ]:
corr = Correlation.PARTIAL_GLASSO.calculate(resp_std, assume_centred=True)
plot_corr_network(resp_std.drop("participant_id", "wave"), corr)
betweenness(corr, resp_std)

# Distance correlation

In [ ]:
corr = Correlation.DISTANCE_CORR.calculate(resp_std, assume_centred=True)
plot_corr_network(resp_std.drop("participant_id", "wave"), corr, threshold=0.3)
betweenness(corr, resp_std, threshold=0.3)

# Contemporaneous network

In [ ]:
corr = Correlation.VAR_CONTEMPORANEOUS.calculate(resp_std)
plot_corr_network(resp_std.drop("participant_id", "wave"), threshold=0.075, corr=corr)
betweenness(corr, resp_std, threshold=0.05)

# Contemporaneous correlation (regularised)

In [ ]:
corr = Correlation.VAR_CONTEMPORANEOUS.calculate(resp_std, regularised=True)
plot_corr_network(resp_std.drop("participant_id", "wave"), threshold=0.05, corr=corr)
betweenness(corr, resp_std, threshold=0.05)

# Temporal network

In [ ]:
corr = Correlation.VAR_TEMPORAL.calculate(resp_std)
# plot_corr_network(resp_std.drop("participant_id", "wave"), threshold=0.04, corr=corr, directed=True)
plot_temporal_network(resp_std.drop("participant_id", "wave"), corr, threshold=0.04)
betweenness(corr, resp_std, threshold=0.04)

In [ ]:
corr = Correlation.VAR_TEMPORAL.calculate(resp_std, regularised=True)
# plot_corr_network(resp_std.drop("participant_id", "wave"), threshold=0.04, corr=corr, directed=True)
plot_temporal_network(resp_std.drop("participant_id", "wave"), corr, threshold=0.01)
betweenness(corr, resp_std, threshold=0.01)

In [ ]:
rng = np.random.default_rng(202602271618)

true_eigs = pa_true_eigs(resp_std)
rand_eigs = pa_random_eigs(resp_std, repeats=100, rng=rng)

fig, ax = plt.subplots(figsize=(4.5, 2), constrained_layout=True)
ax.plot(np.arange(len(true_eigs)), true_eigs, color="tab:blue", label="Data")
ax.plot(np.arange(len(rand_eigs)), rand_eigs, color="tab:red", label="Random")
ax.set_xticks(np.arange(0, len(true_eigs), 2), np.arange(0, len(true_eigs), 2) + 1)

ax.set_xlim(0, 44)
ax.set_ylim(0, None)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("Eigenvalue sorted index (decreasing)")
ax.set_ylabel("Eigenvalue")
ax.legend();

In [ ]:
pca = PCA(n_components=2)
pca.fit(resp_std.to_numpy())

In [ ]:
pca.explained_variance_

In [ ]:
cmap = sns.diverging_palette(20, 230, as_cmap=True)

g = sns.clustermap(
    (pca.components_.T * np.sqrt(pca.explained_variance_)).T,
    center=0,
    cmap=cmap,
    vmin=-1,
    vmax=1,
    method="ward",
    dendrogram_ratio=(0, 0.5),
    cbar_pos=None,
    linewidths=0.75,
    figsize=(4, 2),
    fmt=".1f",
    annot=True,
)

labels = np.asarray(
    resp_std.select(pl.col("cc1", "cc2", "cc3", "cc6", "cc10", "cc11", "cc12")).columns
)
ord_labels = labels[g.dendrogram_col.reordered_ind]

g.ax_heatmap.set_xticks(
    np.arange(pca.components_.shape[1]) + 0.5,
    ord_labels,
    rotation=45,
    horizontalalignment="right",
);

1. General belief about impacts of climate change
2. High when wealthy and community impacted, and world, poor US not future impacted. Low when world, poor future impacts, and wealthy, comm not currently impacted. May reflect belief about increase in impacts over time. Or degree to which impacts expected to grow.
3. High when current and future wealth impacted, and current poor, world not impacted. Alternatively reverse. May reflect belief about distribution of impacts. i.e., wealthy are not/will not be impacted, poor and world are currently impacted (maybe with no expected change, hence zero evs for cc5 analogs).

In [ ]:
X = pca.transform(resp_std.to_numpy())

In [ ]:
sns.relplot(x=X[:, 0], y=X[:, 1])